# SDD编程

## 阶段零：优化项目的提示词 

>交互提示词示例：

```text
请基于当前项目做一次保守的代码质量优化。

开始前请阅读根目录和相关目录下的 AGENTS.md、README.md、docs/spec/MVP 中的需求与设计文档，并检查当前代码、测试和未提交改动。

优化要求：
1. 保持现有功能、接口和数据行为不变，不新增需求范围外的功能。
2. 删除确认无用的重复代码和抽象，简化难以理解的条件、状态和数据流。
3. 优先复用项目已有的函数、组件和工具，保留清晰易读的代码结构。
4. 不要为了统一风格而大范围重写；没有明确收益的代码保持不动。
5. 不要擅自升级或添加依赖，也不要改变数据库结构或接口约定；确有必要时先说明原因和影响。
6. 修改后运行相关测试；如果测试失败，修复由本次改动引起的问题并重新运行。
7. 最后说明改动内容、测试命令和结果，以及尚未验证的事项。
```

## 阶段一：准备阶段

### 1. 创建项目目录

```text
项目名/
├─ backend/        - 后端代码
├─ frontend/       - 前端代码
├─ docs/spec/MVP/  - MVP版本的 spec 开发文档
├─ tests/MVP/      - 测试代码
├─ README.md       
├─ .gitignore
└─ AGENTS.md       
```

将项目目录写入 README.md

### 2. 明确 MVP 版本目标与范围

通过与 AI 交流，明确：
- 解决什么问题
- 实现哪些功能
- 暂时不做什么
- 使用人群

然后将其写入**README.md**

### 3. 明确技术栈

让 AI 通过目前**README.md**的内容推荐技术栈。

>交互提示词示例：

```text
请根据当前 README.md 中的内容，结合本机已有条件，推荐一套适合本项目的技术栈以及版本范围，并简要说明理由。优先复用满足要求的已有环境；确需不同版本时，说明原因及多版本并存方案，避免影响其他项目。信息不足时，只询问影响选型的关键问题。
```

- 确认一下前端框架是否可以用最新版,后续前端的初始化命令都是用的最新版

确认后，将实际采用的技术栈以及版本范围写入**README.md**

#### 3.1 技术栈参考

- 前端：**Next.js** / **React + Vite**
- 后端：**Python + FastAPI**
- 数据库： **SQLite** / **PostgreSQL**
- AI 应用框架：**LangChain**
- Agent 流程编排：**LangGraph**
- AI模型：DeepSeek API （模型版本 deepseek-v4-pro，接口兼容OpenAI调用格式）
- 依赖管理：**uv + npm**
- 版本管理：**Git**

### 4. 确认技术栈所需的环境

让 AI 通过**README.md**技术栈的内容告知需要什么环境然后验证

>交互提示词示例：

```text
根据 README.md 中确定的技术栈及版本范围，检查所需工具是否已安装，
以及版本是否满足要求；缺少或不兼容时替我安装、调整。
```

### 5. 项目初始化

#### 5.1 根目录初始化 Git

在项目根目录运行：git init

#### 5.2A 初始化前端 Next.js

在 frontend 前端文件夹执行：npx create-next-app@latest .

交互选项建议：

```text
TypeScript                             Yes
ESLint                                 Yes
React Compiler                         Yes（如果出现）
Tailwind CSS                           Yes
代码放入 src/ 目录                      Yes
App Router                             Yes
Turbopack                              Yes（如果出现）
Customize the default import alias?    No
```

预览前端页面，在 frontend 前端文件夹执行：

```powershell
npm run dev
```

#### 5.2B 初始化前端 React + Vite

在 frontend 前端文件夹执行：npm create vite@latest . -- --template react

预览前端页面，在 frontend 前端文件夹执行：

```powershell
npm install
npm run dev
```

#### 5.3 初始化后端

在 backend 后端目录执行：

```powershell
uv init
uv add "fastapi[standard]"
```

在`main.py`复制下面的代码：

```python
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware

app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=[
        "http://localhost:5173",  # Vite 默认开发地址
        "http://localhost:3000",  # Next.js 默认开发地址
    ],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.get("/api/health")
def health_check():
    return {"status": "ok"}
```

然后在 backend 下执行：uv run fastapi dev

浏览器打开健康检查接口，能看到：{"status": "ok"}

### 6. 前后端联调

分别打开两个终端。

终端一：

```powershell
cd Zesay/backend
uv run fastapi dev
```

终端二：

```powershell
cd Zesay/frontend
npm run dev
```

浏览器打开前端页面，打开开发者工具，切换到 Console（控制台）
输入下面代码，发送前端测试请求：

```ts
const response = await fetch("http://127.0.0.1:8000/api/health");
const data = await response.json();
console.log(data.status); //输出 ok 表示联调成功
```

### 7. 配置后端地址环境变量后再联调

在 frontend 目录下新建`.env.local`配置后端地址

#### 7.1A next.js框架：

```dotenv
NEXT_PUBLIC_API_URL=http://127.0.0.1:8000
```

导入方式：

1.在`src/app/page.tsx`的第一行添加

```TypeScript
"use client";
```

2.在`export default function Home() { `里面添加

```TypeScript
const apiUrl = process.env.NEXT_PUBLIC_API_URL;
console.log("后端地址：", apiUrl);
```

#### 7.1B Vite 构建的 React 框架：

```dotenv
VITE_API_URL=http://127.0.0.1:8000
```

导入方式：

在`src/App.jsx`文件的`const [count, setCount] = useState(0)`后面加上

```TypeScript
const apiUrl = import.meta.env.VITE_API_URL;
console.log("后端地址：", apiUrl);
```

#### 7.2 检查是否能读到后端地址配置

修改 .env.local 后，重启前端服务，浏览器打开前端页面，打开开发者工具，切换到 Console（控制台），如果显示 http://127.0.0.1:8000，说明环境变量读取成功。

### 8. 配置 .gitignore

```.gitignore
# ==================================================
# 1. 环境变量与敏感配置
# ==================================================

.env
.env.*

# 允许提交配置示例；示例中只能使用占位值
!.env.example
!.env.sample
!.env.template
!.env.*.example

# 专门存放私密配置的目录
/secrets/
/backend/secrets/

# 私钥及证书容器
*.key
*.pem
*.p12
*.pfx


# ==================================================
# 2. Node.js：依赖、缓存与日志
# ==================================================

node_modules/
.npm/
.pnpm-store/

npm-debug.log*
yarn-debug.log*
yarn-error.log*
pnpm-debug.log*


# ==================================================
# 3. 前端构建产物与工具缓存
# ==================================================

# Vite及其他前端构建产物
/frontend/dist/
/frontend/build/

# Next.js
.next/
/frontend/out/
next-env.d.ts

# 开发及构建缓存
.vite/
.turbo/
.eslintcache
.stylelintcache
*.tsbuildinfo

# 部署工具生成的本地信息
.vercel/
.netlify/


# ==================================================
# 4. Python：虚拟环境与编译缓存
# ==================================================

.venv/
venv/
ENV/

__pycache__/
*.py[cod]
*$py.class

# Python包构建与安装产物
/backend/build/
/backend/dist/
*.egg-info/
.eggs/
*.egg
pip-wheel-metadata/

# 检查工具缓存
.pytest_cache/
.mypy_cache/
.ruff_cache/
.pyre/
.pytype/
.tox/
.nox/

# 项目内缓存
.cache/
.uv-cache/


# ==================================================
# 5. 测试与覆盖率生成物
# ==================================================

.coverage
.coverage.*
htmlcov/
coverage/
coverage.xml

# 浏览器自动化测试生成物
playwright-report/
test-results/
blob-report/

# 保留测试代码和手写的验收文档
# 不要忽略 tests/、tests/MVP/ 或 docs/


# ==================================================
# 6. SQLite数据库及运行文件
# ==================================================

*.db
*.sqlite
*.sqlite3

# SQLite事务日志、WAL及共享内存文件
*.db-journal
*.db-wal
*.db-shm
*.sqlite-journal
*.sqlite-wal
*.sqlite-shm
*.sqlite3-journal
*.sqlite3-wal
*.sqlite3-shm


# ==================================================
# 7. 运行时数据、上传资料与备份
# ==================================================

# 这些目录用于运行数据，不用于存放源代码
/data/
/backend/data/

/uploads/
/backend/uploads/

/backups/
/backend/backups/

# PostgreSQL本地数据目录（如果使用）
/pgdata/
/postgres-data/

# 向量数据库本地数据（如果使用）
/chroma_data/
/backend/chroma_data/
/vector_data/
/backend/vector_data/

# 不要统一忽略 *.sql：
# 数据库建表脚本和迁移脚本通常应该提交
# 数据库导出文件应放进 backups/ 等忽略目录


# ==================================================
# 8. 日志、进程与临时文件
# ==================================================

logs/
*.log
*.pid
*.pid.lock

*.tmp
*.temp
*.bak
*.swp
*.swo
*~


# ==================================================
# 9. 编辑器与操作系统生成物
# ==================================================

.vscode/
.idea/
*.iml
*.suo
*.user

.DS_Store
._*
Thumbs.db
ehthumbs.db
Desktop.ini
$RECYCLE.BIN/

# Jupyter自动检查点
.ipynb_checkpoints/
```

## 阶段二：SDD 开发流程

### 1. 需求分析：PRD、用户故事和验收标准

目标：

根据 README.md 当前的内容和 AI 交流得到下面两个文档

- 产品需求文档（PRD.md）：包含产品目标、用户、功能范围、业务规则和非功能需求。
- 用户故事与验收标准（user-stories.md）：每条用户故事附上可验证的验收标准。

完成后要根据用户故事再一次修改 PRD.md

将两个文件放在项目根目录的 /docs/spec/MVP 下

### 2. 原型图设计

目标：

在写代码之前，先用 HTML 画出页面的交互原型，确认最终的效果和交互过程。

>交互提示词示例：

```text
请阅读 README.md、docs/spec/MVP 下的 PRD.md 和 user-stories.md，为当前 MVP 版本设计可交互的页面原型。

要求：
1. 根据需求和用户故事梳理页面、功能入口及主要操作流程，覆盖当前 MVP 范围，不自行增加功能。
2. 使用 HTML、CSS、JavaScript 实现原型，采用虚构数据，暂不连接后端、数据库或真实模型。
3. 主要按钮、页面切换、表单和弹窗可以操作，让我能体验从进入页面到完成任务的完整流程。
4. 按页面需要展示正常、加载、空数据、失败、提交中等状态；仅在需求涉及账号和权限时增加无权限、登录过期状态。
5. 页面清晰易用，布局和组件风格统一。
6. 将原型文件放在 docs/spec/MVP/prototype/，以 index.html 作为统一入口，确保双击即可打开操作，并检查主要交互是否正常。
7. 简要说明页面与用户故事的对应关系，以及需要我体验确认的重点。

信息不足时，只询问影响主要流程或布局的关键问题，其余采用合理默认方案并说明。
```

生成初始原型图后，用 VibeCoding 的方式改造原型图。

### 3. 业务设计：把需求和原型变成技术方案

目标：

将需求和原型转化为可执行的技术方案。

产出：

1. 项目宪法（根目录下的 AGENTS.md 还有MVP下的 AGENTS.md）

1. 业务术语表（glossary.md）

1. 数据库设计（db_design.md）

1. 接口设计（api_design.md）

#### 3.1 编写项目宪法（AGENTS.md）

##### 3.1.1 根目录下的 AGENTS.md

>交互提示词示例：

```text
请为"项目名"项目生成根目录下的 AGENTS.md，内容包括：

技术栈：参考 README.md 里的技术栈

目录结构：参考 README.md 里的目录结构

业务接口响应格式：统一为 {code, message, data}。初始化健康检查 /api/health 为明确例外，返回 {"status":"ok"}。

请生成完整的 AGENTS.md 文件，放在项目根目录下
```

##### 3.1.2 MVP目录下的 AGENTS.md

>交互提示词示例：

```text
请为"项目名"项目生成 MVP 版本的 AGENTS.md，内容包括：

约束条件：……

代码风格：……

验收相关约束：……

MVP 版本全部产出文档清单：用来存放 docs/spec/MVP/ 下的所有文档

请生成完整的 AGENTS.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.2 业务术语表（Glossary）

>交互提示词示例：

```text
请根据需求文档和用户故事生成业务术语表。

格式：术语 | 定义

请生成完整的 glossary.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.3 数据库设计

>交互提示词示例：

```text
请根据 docs/spec/MVP 下的现有文档，设计"项目名"的数据库结构。

请生成完整的数据库设计文档，包含：
- 每张表的字段、类型、约束、描述
- 如果还需要其他表，你要提示我
- 明确主键、外键、表之间的关系、必要索引、数据归属和删除规则

生成完整的 db_design.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.4 API接口设计

>交互提示词示例：

```text
请根据 docs/spec/MVP 下的现有文档，生成"项目名"当前 MVP 范围内的接口

需要注意的是：
1. 接口名不要太复杂，简单一些即可
2. 仅设计当前 MVP 范围内的接口，覆盖原型中的实际操作，不要遗漏
3. 明确请求方法、路径、参数及校验、成功与错误响应、HTTP 状态码、业务 code、认证方式和数据访问权限

请生成完整的 api_design.md 文件，放在项目根目录的 /docs/spec/MVP 下
```

#### 3.5 添加文档清单

将 MVP 版本产出的文档清单添加到 /docs/spec/MVP/AGENTS.md 的“MVP 版本全部产出文档清单”中，应该包括：

- PRD.md
- user-stories.md
- prototype/
- glossary.md
- db_design.md
- api_design.md
- SDD-开发单元拆解.md # 第 4 部分生成

### 4. 编码

#### 4.1 SDD开发单元拆解

>交互提示词示例：

```text
请阅读项目根目录下的 AGENTS.md 以及 docs/spec/MVP 下的所有文档，同时检查现有代码和已完成的开发单元；项目尚未初始化时，按新项目处理。把 MVP 阶段拆成适合在 SDD 开发中逐次交给 AI 完成的开发单元。

一个开发单元必须围绕一个明确目标，按本开发单元的需要，在一次开发中完成涉及的前端、后端、数据库、AI 调用、权限处理、异常状态和测试，并且完成后可以独立运行、演示和验收。业务开发单元应形成用户操作到结果展示的完整闭环；基础能力单元应形成自身的实现、测试和验收闭环。

不要只按页面或 API 拆分，也不要把多个无关功能合并。如果一个单元过大，请继续拆分；如果多个小任务必须一起才能形成完整功能，请合并。请按照前置依赖安排顺序。

输出以下内容：

1. 开发单元编号和名称；
2. 主要目标；
3. 前置依赖；
4. 本次包含的范围；
5. 本次明确不包含的范围；
6. 预计涉及的页面、接口和数据；
7. 验收标准；
8. 测试方式；
9. 优先级和复杂度。

请使用具体名称，不要使用“功能一”“功能二”等占位名称，并指出哪些内容属于基础能力、业务功能或公共能力。

生成 SDD-开发单元拆解.md 文件放在项目根目录的 /docs/spec/MVP 下
```

#### 4.2 为开发单元生成执行提示词

>交互提示词示例：

```text
请阅读项目根目录下的 AGENTS.md、docs/spec/MVP 下的所有文档，同时检查现有代码和已完成的开发单元；项目尚未初始化时，按新项目处理。为指定的开发单元生成一份可以直接交给编程 AI 执行的开发提示词。

指定开发单元：
【填写开发单元编号和名称】

生成的提示词必须包含：

1. 开发目标；
2. 本次必须完成的内容；
3. 本次明确不处理的内容；
4. 需要先阅读的项目文件；
5. 涉及的前端、后端、数据库和 AI 服务；
6. 接口、数据保存和权限要求；
7. 加载、空数据、失败、超时和重复提交处理；
8. 测试要求：要求执行者将测试代码放在项目根目录的 tests/MVP/ 下，运行相关测试并修复问题；
9. 验收标准：明确本开发单元满足哪些可验证的条件才算完成；
10. 汇报要求：要求执行者在完成后汇报实现内容、测试命令与实际结果、验收情况，以及未完成或未验证的事项。

请严格限制在指定开发单元范围内。如果发现文档冲突、前置依赖未完成或需求信息不足，请在提示词中明确列出，不要擅自扩大范围。

每个开发单元都应尽量形成完整闭环：用户操作、前端交互、后端处理、数据保存、结果展示、异常处理和测试。
只生成开发提示词，不要直接写代码。
```

#### 4.3 实现开发单元

>交互提示词示例：

```text
请严格按照下面的开发任务提示词完成本开发单元：

【本单元的执行提示词】
```

### 5. MVP整体联调、回归测试与验收

目标：

验证各开发单元组合后是否按预期工作，确保当前版本符合需求文档、用户故事和验收标准。

>交互提示词示例：

```text
请结合当前代码、已有测试和 docs/spec/MVP 下的文档，对当前版本进行整体联调、回归测试与验收。

运行已有测试，并补充缺失的跨功能流程测试；
新增测试代码放在项目根目录的 tests/MVP/ 下。

使用独立测试环境或测试数据库，只清理本次测试创建的数据。

修复发现的问题，并重新运行受影响的测试。
对未实现的需求、文档冲突和无法验证的事项，列出清单及原因；
不要自行增加 MVP 范围之外的功能。

最后汇报测试命令、实际结果，以及各项验收标准的满足情况。
```

### 6. 本地运行

目标：

验证项目能够按文档在本地运行。

>交互提示词示例：

```text
请根据项目实际配置，完善 README.md 中的运行说明，包括：

1. 依赖安装；
2. 环境变量配置；
3. 数据库初始化；
4. 本地前后端启动命令、访问地址和停止方式；

实际验证本地启动流程。
如发现启动问题，请排查并修复。
将需要长期遵守的开发注意事项补充到 AGENTS.md，
将面向使用者的运行说明补充到 README.md。
```

运行成功后为每个.env.*文件，生成对应的 .env.\*.example 文件

### 7. 提交到 GitHub（推荐 HTTPS）

#### 7.1 GitHub 建立新仓库

登陆 GitHub 建立新仓库( new repository )

#### 7.2 复制 HTTPS 地址

```text
https://github.com/你的用户名/项目名.git
```

### 7.3 关联本地仓库

```bash
git remote add origin https://github.com/你的用户名/项目名.git
git remote -v  # 查看远程仓库
```

### 7.4 第一次推送

先用 `git branch` 确认分支名。当前分支为 `main` 时执行：

```bash
git branch
git push -u origin main

git push    # 后续只需要这样
```

## 阶段三：服务器部署与上线

### 1. 准备云服务器

云服务器平台示例：阿里云、百度云。

需要准备：
- 云服务器或轻量应用服务器，操作系统选择 Ubuntu 24.04 LTS。
- 要有公网 IP。

进入云平台的安全组或防火墙页面，配置入站连接：

| TCP 端口 | 用途 |
| --- | --- |
| 22 | SSH 登录 |
| 80 | HTTP 访问及常见证书验证 |

### 2. 登录服务器、准备部署用户

#### 2.1 连接服务器

在本机终端输入：

```bash
ssh 登录用户名@服务器公网IP
```

使用云平台提供的账号和认证方式登录

#### 2.2 准备普通部署用户

如果已有具备 sudo 权限的普通用户，可以直接使用。如果当前以 root 登录，需要创建普通用户时执行：

```bash
adduser 部署用户名          # 创建用户
usermod -aG sudo 部署用户名 # 给用户添加 sudo 权限
su - 部署用户名             # 切换用户
```

#### 2.3 解决 SSH 因空闲自动断连

在目录 `C:\Users\你的Windows用户名\.ssh\config` 添加以下内容，文件不存在时新建 `config`文件（无后缀）

```text
Host *
  ServerAliveInterval 15
  ServerAliveCountMax 3
```

客户端每 15 秒发送一次保活消息，连续 3 次未收到回复时断开。

### 3. 安装所需工具

#### 3.1 安装基础工具和 Nginx

在服务器终端执行：

```bash
sudo apt update
sudo apt install -y git curl ca-certificates nginx
sudo systemctl enable --now nginx
systemctl status nginx --no-pager
nginx -v
git --version
```

`enable --now` 表示立即启动，并设置服务器开机后自动启动。

#### 3.2 安装其他工具

询问 AI 为服务器安装其他工具

>交互提示词示例：

```text
请根据项目 README.md 中确定的技术栈和依赖版本，为 Ubuntu 24.04 LTS 服务器提供环境安装步骤。

服务器通过 SSH 操作，我会手动执行你给出的命令。

1. 先给出检查所需工具的命令。
2. 我返回检查结果后，只为缺少或版本不兼容的工具提供安装命令，优先使用官方安装方式。
3. 标明每条命令的执行位置、用途，以及是否需要 sudo。
4. 安装后给出版本和可执行文件路径的检查命令，供后续配置 systemd 使用。

尽量复用已有环境，避免影响其他项目。
```

### 4. 克隆项目并安装依赖

#### 4.1 配置 GitHub 账号 SSH 密钥

（Deploy Key 作为以后生产部署的参考）

以部署用户登录服务器，先检查是否已有密钥：

```bash
ls -la ~/.ssh
```

目录不存在属于正常情况。需要创建新密钥时执行：

```bash
ssh-keygen -t ed25519 -C "你的邮箱"
```

直接回车选择默认选择，查看默认路径下的公钥：

```bash
cat ~/.ssh/id_ed25519.pub
```

复制以 `ssh-ed25519` 开头的完整公钥，在 GitHub 账号中进入：

```text
GitHub → Settings → SSH and GPG keys → New SSH key
```

填写标题，添加公钥。

验证连接：

```bash
ssh -T git@github.com
```

#### 4.2 克隆项目

在服务器执行：

```bash
cd ~
git clone git@github.com:你的用户名/项目名.git
cd 项目名
```

#### 4.3 按照锁文件安装项目依赖

从项目根目录开始：

```bash
cd frontend
npm ci
cd ../backend
uv sync --locked
cd ..
```

### 5. 配置生产环境变量、数据库和数据目录

#### 5.1 准备生产配置

配置所有`.env*example`文件

#### 5.2 配置前端生产后端地址

这份笔记采用同一域名访问前端和 API，由 Nginx 将 `/api/` 请求转发到 FastAPI。

如果项目沿用准备阶段的写法，即把 `API_URL` 与 `/api/...` 拼接，则可以在服务器 `frontend/.env.production` 中配置网站的完整来源地址，不附加 `/api` 或末尾斜杠。

React + Vite：

```dotenv
VITE_API_URL=https://你的域名
```

Next.js：

```dotenv
NEXT_PUBLIC_API_URL=https://你的域名
```

项目也可以统一使用 `/api/...` 相对路径；是否需要上述变量，以实际请求代码为准，避免重复拼接 `/api`。暂时通过公网 IP 验证 HTTP 时，可先使用 `http://服务器公网IP`，切换 HTTPS 域名后再更新配置并重新构建。

浏览器中的 `127.0.0.1` 指向访问者自己的电脑，不能把本地后端地址原样用于正式网站。`VITE_` 和 `NEXT_PUBLIC_` 开头的变量会提供给浏览器，不要放模型 API 密钥等秘密。

生产地址应在前端构建前配置；Vite 和 Next.js 的浏览器公开环境变量通常会写入构建产物，修改后需要重新构建并发布。不要把本地调试用的 `.env.local` 原样复制到服务器，以免覆盖生产配置。

#### 5.3 准备数据库和持久化目录

- SQLite：确定数据库文件的绝对路径，并让后端服务用户能够读写数据库所在目录。
- PostgreSQL：准备数据库、应用账号和连接配置，按设计授予权限。
- 为数据库文件、上传资料和备份准备持久化目录，避免放进会被构建、清理或替换的代码发布目录。
- 首次部署执行项目规定的初始化或迁移命令；已有数据时先备份，再执行必要的迁移，不重复执行会清空数据的初始化。

>交互提示词示例：

```text
请检查当前项目代码、README.md 和数据库设计，整理服务器生产配置与数据准备步骤。

列出实际需要的环境变量、用途、保存位置和加载方式；区分前端公开配置与后端私密配置，提供不含真实密钥的示例。检查前端 API 地址和 /api 路径的拼接方式，给出与 Nginx 同域转发匹配的配置。

根据当前数据库方案，给出数据库创建、账号权限、初始化或迁移命令，以及数据库、上传文件、备份的实际目录和文件权限。说明命令的执行目录，并区分首次部署和已有数据的情况。

确认后端服务运行用户能够访问所需数据。把不含秘密的操作说明记录到 docs/deployment.md。
```

### 6. 构建前端，启动并管理应用服务

目标：生产服务在 SSH 断开后继续运行，发生进程退出时可按配置恢复，并在服务器重启后自动启动。

#### 6.1A React + Vite

在服务器的 `frontend` 目录执行：

```bash
npm run build
```

默认构建产物在 `frontend/dist/`。第 7 步将其发布到静态网站目录，例如 `/var/www/项目名/`，由 Nginx 提供访问。若项目修改了输出目录，以实际配置为准。

Vite 的这种部署方式不需要常驻 Node 服务；`npm run dev` 和 `npm run preview` 不用作本笔记的生产网站服务。

#### 6.1B Next.js

先根据项目实际功能确定部署模式。

需要 Next.js 服务端能力时，在 `frontend` 目录构建：

```bash
npm run build
```

如果 `package.json` 中的 `start` 脚本是 `next start`，可以用下面的命令临时检查启动：

```bash
npm run start -- --hostname 127.0.0.1 --port 3000
```

正式运行交给第 6.3 步的 systemd 管理，再由 Nginx 转发请求。检查结束后按 `Ctrl+C` 停止临时进程，避免与正式服务争用端口。项目如果采用 standalone 输出，按该模式的实际入口和文件布局部署。

只有项目满足静态导出条件、且已配置静态导出时，才按静态文件部署；默认导出目录通常为 `out/`，该模式无需启动 Next.js 服务。

#### 6.2 启动 FastAPI 后端

以下示例适用于后端入口仍为 `backend/main.py`、应用变量为 `app` 的项目。在 `backend` 目录临时检查：

```bash
uv run --locked fastapi run main.py --host 127.0.0.1 --port 8000
```

入口发生变化时，以实际项目结构为准。确保当前启动方式已经加载第 5 步的后端配置；最终由 systemd 统一管理配置和启动。

在另一个服务器终端检查：

```bash
curl -f http://127.0.0.1:8000/api/health
```

应该返回 `{"status":"ok"}`。检查结束后按 `Ctrl+C` 停止临时进程；正式服务不使用 `fastapi dev` 或自动重载模式。

#### 6.3 配置 systemd 服务

FastAPI 需要常驻服务；采用 Node 服务模式的 Next.js 也需要。Vite 静态前端不需要创建 Node 服务。

>交互提示词示例：

```text
请根据当前项目和 Ubuntu 24.04 LTS 服务器环境，为应用生成可实际使用的生产构建、启动和 systemd 配置。

部署用户：【实际用户名】
项目绝对路径：【实际路径】
前端部署模式：【Vite 静态文件 / Next.js Node 服务 / Next.js 静态导出】
生产配置和数据目录：【第 5 步确认的路径】

要求：
1. 检查 package.json、后端入口和真实可执行文件路径，给出正确的构建与生产启动命令。
2. 为 FastAPI，以及确实需要的 Next.js Node 服务分别生成 systemd unit；设置 User、WorkingDirectory、ExecStart、实际配置文件的加载方式、失败后重启和开机启动。
3. ExecStart 使用明确的可执行文件路径，不依赖交互式终端的 PATH、手动激活虚拟环境或登录脚本；后端可直接使用项目 .venv 中的生产服务程序。
4. FastAPI 监听 127.0.0.1:8000，Next.js Node 服务监听 127.0.0.1:3000；端口已占用时先查明原因，再确定实际配置。
5. 说明文件保存位置、权限及各项配置含义，给出 daemon-reload、enable --now、status、restart、stop 和 journalctl 查看日志的完整命令，使用实际服务名。
6. 给出服务器内部的健康检查方法。检查完成后记录到 docs/deployment.md。
```

### 7. 配置 Nginx、域名和 HTTPS

#### 7.1 确定访问方式

- React + Vite：Nginx 提供构建后的静态文件；使用浏览器路径路由时，配置前端页面的回退规则。
- Next.js Node 服务：Nginx 将前端请求转发到本机 Next.js 服务。
- Next.js 静态导出：Nginx 提供导出的静态文件，并按实际导出目录结构处理路由。
- `/api/` 请求：转发到 FastAPI。当前后端路由包含 `/api` 前缀，例如 `/api/health`，因此转发时必须保留该前缀。

静态文件优先发布到 `/var/www/项目名/` 等专用目录，只给 Nginx 必需的目录访问和文件读取权限。应用代码、数据库和私密配置不作为静态文件公开。不统一修改整个家目录的权限，也不使用 `chmod 777`。

#### 7.2 配置 HTTP 和 Nginx

>交互提示词示例：

```text
请根据当前项目和服务器上的实际部署结果，教我一步步配置 Nginx。

服务器公网 IP：【填写】
域名：【填写；暂时没有则说明】
前端部署模式和构建产物位置：【填写】
FastAPI 内部地址：【例如 http://127.0.0.1:8000】
Next.js 内部地址：【仅 Node 服务模式填写】

请给出：
1. 需要创建或修改的配置文件绝对路径，说明现有配置的含义、修改内容及原因；保留服务器上其他网站的配置。
2. 与当前前端模式匹配的站点配置和发布命令；静态文件放在专用网站目录，处理实际需要的前端路由和静态资源访问。
3. /api/ 到 FastAPI 的转发配置。先检查后端路由，再确定 proxy_pass 写法，避免把 /api/health 错转成 /health；说明请求路径的实际变化。
4. 正确传递 Host、客户端地址和协议相关头部；检查后端对代理头的信任设置，仅信任实际代理。项目涉及流式 AI 输出或 WebSocket 时，配置相应的缓冲、超时或连接升级规则。
5. 配置文件启用方式，以及 nginx -t 校验、重载和排错日志命令。
6. 验证首页、静态资源、页面刷新和 /api/health 的方法。

如果前后端确实使用不同来源，再按实际生产地址配置 CORS。把最终方案记录到 docs/deployment.md。
```

修改后先校验，通过后再重载：

```bash
sudo nginx -t
sudo systemctl reload nginx
```

#### 7.3 配置域名和 HTTPS

在域名服务商处把域名解析到服务器公网 IP；检查已设置的 A/AAAA 记录是否都指向可用服务器。确认 80、443 端口能访问；按所选云服务区域和服务商要求完成必要的域名接入或备案流程。

本笔记按“域名解析 + Nginx + ACME 证书”的常见方式配置 HTTPS，具体安装命令以服务器当前环境为准。

>交互提示词示例：

```text
当前 Nginx 的 HTTP 站点已验证。请基于 Ubuntu 24.04 LTS 和现有 Nginx 配置，给出为【实际域名】配置 HTTPS 的操作步骤。

检查域名解析和端口可达性，选择与当前环境兼容的 Certbot 安装方式，说明证书申请、Nginx 配置、HTTP 跳转 HTTPS、自动续期及续期测试命令。

如果项目有登录会话，检查 HTTPS 下的 Cookie 配置；验证浏览器不再请求旧的 HTTP 或 localhost 地址。前端公开环境变量发生变化时，重新构建并发布；需要运行 Next.js 服务时重启对应服务。

记录证书位置、续期方式、实际验证结果和未完成事项到 docs/deployment.md。
```

证书配置完成后，可以执行续期演练，并确认自动续期任务已启用：

```bash
sudo certbot renew --dry-run
```

### 8. 线上验收

目标：确认用户通过真实网站地址可以完成主要任务，且数据和应用服务在重启后仍然正常。

至少验证：
- 从本机浏览器或另一台设备，通过最终 HTTPS 地址访问网站。
- 首页、静态资源、主要页面切换和页面刷新正常。
- `/api/health` 正常，实际业务接口可以调用。
- 按用户故事完成主要流程；真实模型调用、流式输出或上传等能力按项目范围验证。
- 新增和修改的数据可以再次读取，刷新页面和重启服务后仍然存在。
- 项目要求的登录、权限和数据隔离按验收标准工作。
- SSH 断开后应用仍运行；首次上线的可安排时段内，重启服务器并确认服务自动恢复、数据仍在。
- 生产构建、前后端日志和 Nginx 日志中没有未处理的部署错误。

使用可识别的测试数据，测试后只清理本次创建的数据。已有用户使用的网站，重启等操作安排在维护时段。

>交互提示词示例：

```text
请根据 docs/spec/MVP 中的用户故事与验收标准，以及 docs/deployment.md，对已部署的【实际网站地址】进行线上验收。

检查页面与 API、主要业务流程、真实 AI 服务、异常处理、数据持久化和需求中规定的访问权限，并验证 SSH 断开后服务持续运行。根据当前网站使用情况安排服务及服务器重启验证，确认自动启动和数据保留。

使用本次创建的测试数据，避免修改已有真实业务数据。记录检查方法、实际结果和日志线索；无法直接访问服务器或验证的部分，给我具体执行步骤，不要把未执行的检查写成通过。

发现问题时在当前 MVP 范围内修复，更新受影响的服务或构建产物，并重新验证相关流程。
把验收结果和未解决事项记录在 docs/deployment.md，确认是否达到当前版本的上线要求。
```

### 9. 更新发布、备份恢复和回退

#### 9.1 后续更新发布

先在开发环境完成修改和验收，再提交推送。服务器更新前记录当前代码版本、配置和可用构建产物，并备份受影响的数据。

在服务器项目目录内检查和拉取当前发布分支：

```bash
git status --short
git rev-parse HEAD
git pull --ff-only
```

如果有未提交改动、分支分歧，或当前采用标签/提交号部署，先按实际发布方式处理，不直接覆盖或强行拉取。

更新的实际顺序通常为：

1. 记录旧版本并备份，明确本次是否需要维护时段。
2. 更新到已验收的代码，按锁文件同步所需依赖。
3. 根据新版本要求准备配置；按迁移方案执行必要的数据库变更。
4. 重新构建前端，并发布新的静态文件或重启 Next.js 服务。
5. 重启受影响的后端服务；Nginx 配置有变化时先校验再重载。
6. 验证健康检查和主要业务流程，记录本次部署的提交号与结果。

代码、数据库和前端产物必须匹配同一发布版本。具体停机、迁移和切换顺序按项目兼容性确定；不要在数据库结构尚未就绪时切换不兼容的新代码。

#### 9.2 备份与恢复

按项目实际情况备份数据库、上传文件和恢复运行所需的配置。备份频率、保留时间和保存位置根据能接受的数据损失确定；至少保留一份服务器之外的受保护副本。

- SQLite：使用 SQLite 支持的备份方式，或按停写方案备份完整数据库；运行中的数据库尤其是 WAL 模式，不只随手复制一个 `.db` 文件。
- PostgreSQL：按恢复要求选择 `pg_dump` 等备份方式，并记录对应的恢复命令。
- 在独立数据库或隔离目录进行恢复演练，确认数据可读取、应用可运行，不覆盖正在使用的数据。

#### 9.3 发布失败时回退

保留可用的上一版本代码、依赖锁文件、构建产物和对应配置。发布失败时，按已记录的步骤恢复应用版本并重新验收。

如果新版本变更了数据库结构，需要先检查旧代码是否兼容；代码回退不会自动回退数据库。恢复旧数据库备份可能丢失备份之后的新数据，必须明确恢复范围和处理办法。

>交互提示词示例：

```text
请根据当前服务器部署方式，完善 docs/deployment.md 中的更新发布、备份恢复与失败回退流程。

要求：
1. 使用实际目录、服务名和命令，写清版本记录、数据备份、代码更新、锁文件依赖同步、环境配置、数据库迁移、前端构建与发布、服务重启及验收顺序。
2. 给出适合当前数据库和上传目录的备份方案，包括执行频率、保留时间、服务器之外的保存位置、访问权限和结果检查方式；涉及需要我提供的存储信息时明确列出。
3. 给出在隔离环境恢复备份并验证的步骤，确认备份实际可用。
4. 给出回退到上一可用版本的方法，区分应用回退与数据库恢复，并说明如何处理发布后的新增数据。
5. 记录日志位置和常用查看命令，给出适合当前规模的服务异常、磁盘空间和备份失败检查方式。
6. README.md 中添加部署文档入口；文档只记录操作方法和配置占位值，不记录真实密钥。

标明已实际验证和仍未验证的部分。不要把仅生成了命令当作已经完成发布或恢复演练。
```

完成标准：线上主要功能已经验收，服务能持续运行并在重启后恢复，数据持久化正常，备份恢复已验证，而且下一次更新和失败回退有可执行的说明。